# 13b - Follow-up: where the model fails on new field photos, and a threshold that can be trusted

**This is a follow-up to notebook 13, added after seeing its results, and reported as such.**

**Why:** notebook 13 kept ResNet-50 (D2) and temperature scaling (D6, T = 3.04), but on the
selection set ResNet-50 is right only 57% of the time with the crop given, and the pre-set
threshold rule (D7: lowest threshold with at least 90% accuracy on answered photos) was met
by just **2 of 3,821 photos**, giving 0.945 with a bootstrap range of 0.89-1.0. A
threshold set by 2 photos is noise.

**Rules fixed before this notebook was run (18 Oct):**

1. **Threshold (D7, clarified):** the lowest threshold whose **bootstrap lower bound** of
   accuracy on answered photos is at least 90% (1,000 resamples; a resample with no answered
   photo counts as 0). If no threshold reaches that, the **pre-set fallback** applies: the
   cost-based threshold, where a wrong answer costs as much as 5 referrals. The original
   rule's 0.945 is reported alongside. This is a clarification of a rule that broke down
   with very few photos, not a new target: 90% stays.
2. **No source is dropped because its accuracy is low.** A source could only be removed if
   its labels were shown to be wrong by looking at the photos.
3. **Every wrong-answer rate is reported with how many photos were referred.**
4. **NFR i** (calibration on in-domain photos) is reported as it is, not redefined.

Uses the scores notebook 13 saved; no GPU needed. About 5 minutes.

In [1]:
from pathlib import Path
import sys

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/BilliartMuthoni/confidence-calibrated-crop-diagnosis.git"
REPO_DIR = Path("/content/drive/MyDrive/Sproutly-app")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

ML_DIR = REPO_DIR / "ml"
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

import config

Mounted at /content/drive


In [2]:
!git -C {REPO_DIR} config core.fileMode false
!git -C {REPO_DIR} pull --ff-only
!git -C {REPO_DIR} status --short --branch

Already up to date.
Refresh index: 100% (242/242), done.
## main...origin/main
 M ml/notebooks/13b_threshold_followup.ipynb
?? ml/results/fusion_quick_test/


In [3]:
%pip install -q -r {ML_DIR}/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.9/373.9 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.1/405.1 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 104.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 4.4 MB/s eta 0:00:00


## Load the scores saved by notebook 13 and rebuild its calibration

In [ ]:
import importlib
import json

import numpy as np
import pandas as pd

from src import calibration as cal, metrics, training
for module in (cal, metrics, training):
    importlib.reload(module)

TARGET, COST_WRONG, FLOOR, BOOTSTRAP = 0.90, 5.0, 0.5, 1000
RUN = config.MODELS_DIR / "compare_resnet50"
N13 = config.MODELS_DIR / "calibration"
OUT_DIR = config.MODELS_DIR / "calibration_followup"
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPO_RESULTS = ML_DIR / "results" / "calibration_followup"
REPO_RESULTS.mkdir(parents=True, exist_ok=True)

CLASSES = list(config.PLANTVILLAGE_CLASSES)
INDEX = {name: i for i, name in enumerate(CLASSES)}
CLASS_CROPS = np.array([{"Corn_(maize)": "maize", "Tomato": "tomato", "Potato": "potato"}[c.split("___")[0]]
                        for c in CLASSES])
SHORT = {c: c.replace("Corn_(maize)___", "Maize ").replace("___", " ").replace("_", " ") for c in CLASSES}

def part(rows, logits):
    labels = rows["class"].map(INDEX)
    keep = labels.notna().to_numpy()
    labels = labels[keep].astype(int).to_numpy()
    return {"rows": rows[keep].reset_index(drop=True), "labels": labels, "logits": logits[keep], "crops": CLASS_CROPS[labels]}

sets = {}
for name in ["test", "ood_test"]:
    sets[name] = part(pd.read_csv(RUN / "results" / f"{name}_predictions.csv"),
                      np.load(RUN / "results" / f"{name}_predictions.logits.npy"))
sets["selection"] = part(pd.read_csv(config.SPLITS_DIR / "selection.csv"), np.load(N13 / "resnet50_selection.logits.npy"))
sel = sets["selection"]

FINAL = cal.fit("temperature", sel["logits"], sel["labels"])
oof = cal.out_of_fold("temperature", sel["logits"], sel["labels"], 5, config.RANDOM_SEED)
saved = json.loads((N13 / "calibrator.json").read_text())
assert abs(FINAL.params["temperature"] - saved["params"]["temperature"]) < 1e-6, "not the calibration notebook 13 chose"
print(f"Temperature {FINAL.params['temperature']:.4f} (same as notebook 13)")

def app(probs, p):
    conf, predicted = cal.crop_confidence(probs, CLASS_CROPS, p["crops"])
    return conf, predicted, predicted == p["labels"]

raw = {name: app(metrics.softmax(p["logits"]), p) for name, p in sets.items()}
calibrated = {name: app(FINAL.probabilities(p["logits"]), p) for name, p in sets.items()}
calibrated["selection"] = app(oof, sel)  # out of fold: no photo scored by a calibrator that saw it

## D7, clarified: a threshold the data can actually support

In [ ]:
conf_oof, _, correct_oof = calibrated["selection"]
D7 = cal.choose_threshold_verified(conf_oof, correct_oof, TARGET, COST_WRONG, FLOOR, n=BOOTSTRAP, seed=config.RANDOM_SEED)
THRESHOLD = D7["threshold"]
D7["answered_at_original_rule"] = int((conf_oof >= D7["original_rule_threshold"]).sum())
print(json.dumps(D7, indent=1, default=float))

## Where the model fails: each source and each class of the selection set

Accuracy with the crop given, confidence before and after calibration, the share answered
at the chosen threshold, and the most common wrong answer. Nothing here is used to drop a
source.

In [ ]:
conf_raw, _, _ = raw["selection"]
_, predicted, correct = calibrated["selection"]
table = sel["rows"][["source", "class"]].assign(
    correct=correct, confidence_before=conf_raw, confidence_after=conf_oof, answered=conf_oof >= THRESHOLD,
    predicted=[SHORT[CLASSES[i]] for i in predicted])

def summary(group):
    wrong = group.loc[~group["correct"], "predicted"]
    return pd.Series({"photos": len(group), "accuracy": group["correct"].mean(),
                      "mean confidence before": group["confidence_before"].mean(),
                      "mean confidence after": group["confidence_after"].mean(),
                      "share answered": group["answered"].mean(),
                      "most common wrong answer": wrong.mode().iat[0] if len(wrong) else "-"})

by_source = table.groupby("source").apply(summary, include_groups=False)
by_class = table.assign(cls=table["class"].map(SHORT)).groupby("cls").apply(summary, include_groups=False)
by_source_class = table.assign(cls=table["class"].map(SHORT)).groupby(["source", "cls"]).apply(summary, include_groups=False)
display(by_source.round(3))
display(by_class.round(3))
display(by_source_class.round(3))
by_source.to_csv(OUT_DIR / "selection_by_source.csv")
by_class.to_csv(OUT_DIR / "selection_by_class.csv")
by_source_class.to_csv(OUT_DIR / "selection_by_source_and_class.csv")

## What each option would mean in the app

Every option on every set, always with how many photos would be referred. A is the
clarified rule (or its fallback), C the original rule's output, B a lower 70% target shown
only for comparison (rejected: it would change the 90% promise after seeing results). The
last row is what the backend does today: uncalibrated confidence and a provisional 0.90.

In [ ]:
options = {
    "A: clarified rule": ("calibrated", THRESHOLD),
    "C: original rule": ("calibrated", D7["original_rule_threshold"]),
    "B: 70% target (rejected)": ("calibrated", cal.threshold_for_accuracy(conf_oof, correct_oof, 0.70, FLOOR)),
    "today: uncalibrated, 0.90": ("raw", 0.90),
}
labels = {"selection": "selection set (new field sources)", "test": "test (in-domain)", "ood_test": "PlantDoc"}
rows = []
for option, (kind, threshold) in options.items():
    for name in ["selection", "test", "ood_test"]:
        conf, _, ok = (calibrated if kind == "calibrated" else raw)[name]
        at = cal.at_threshold(conf, ok, threshold)
        rows.append({"option": option, "threshold": threshold, "set": labels[name], "referred": at["referred"],
                     "answered": at["coverage"], "accuracy of answered": at["selective_accuracy"],
                     "wrong answers given (share of all photos)": at["wrong_and_answered"]})
options_table = pd.DataFrame(rows).set_index(["option", "set"])
display(options_table.round(4))
options_table.to_csv(OUT_DIR / "threshold_options.csv")

## NFR i: calibration on in-domain and out-of-domain photos, as measured

In [ ]:
nfr_rows = []
for name in ["test", "ood_test", "selection"]:
    for kind, source in [("before", raw), ("after (temperature)", calibrated)]:
        conf, _, ok = source[name]
        nfr_rows.append({"set": labels[name], "calibration": kind, "ece": metrics.expected_calibration_error(conf, ok),
                         "mean confidence": conf.mean(), "accuracy": ok.mean()})
nfr = pd.DataFrame(nfr_rows).set_index(["set", "calibration"])
display(nfr.round(4))
nfr.to_csv(OUT_DIR / "nfr_i_calibration.csv")

## Save the decision and the final calibrator for the backend

In [ ]:
final = {"model": "resnet50", "method": "temperature", "temperature": FINAL.params["temperature"],
         "threshold": THRESHOLD, "rule": D7["rule"], "fitted_on": f"selection set ({len(sel['labels'])} photos)",
         "confidence": "best class of the farmer's crop, 11-class probability, not rescaled"}
training.save_json(final, OUT_DIR / "calibrator_final.json")
training.save_json({"D7_clarified": D7, "post_hoc_followup": True}, OUT_DIR / "decision.json")
print(json.dumps(final, indent=1, default=float))

## Copy the results into the repository, sync Drive, save to GitHub

Save the notebook first (Ctrl+S).

In [ ]:
import shutil

for file in OUT_DIR.glob("*"):
    if file.suffix in {".csv", ".json"}:
        shutil.copy(file, REPO_RESULTS / file.name)
print(sorted(p.name for p in REPO_RESULTS.iterdir()))
drive.flush_and_unmount()
drive.mount("/content/drive")

In [ ]:
from google.colab import userdata

from src import github_sync

shutil.copy(ML_DIR / "notebooks" / "13b_threshold_followup.ipynb", REPO_RESULTS / "notebook.ipynb")
print(github_sync.push(
    REPO_DIR,
    branch="colab/threshold-followup",
    message="Follow-up: selection-set failures by source and class, and a verified threshold",
    token=userdata.get("GITHUB_TOKEN"),
    name="BilliartMuthoni",
    email=userdata.get("GIT_EMAIL"),
    paths=[str(REPO_RESULTS.relative_to(REPO_DIR))],
))